### Metalabelling testing

In [1]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
import xgboost as xgb
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PRIMARY_THRESHOLD = 0.55
META_THRESHOLD = 0.55
META_TRAIN_FRAC = 0.30

def plot_cm(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["No Long", "Long"],
        yticklabels=["No Long", "Long"],
    )
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    plt.show()

    print(title)
    print(classification_report(y_true, y_pred, zero_division=0))


def walk_forward_pca_metalabel(
    df_model,
    feature_cols,
    target_col,
    train_size=5000,
    test_size=2500,
    embargo=20,
):
    all_rows = []
    fold_reports = []

    n = len(df_model)
    fold = 0

    for start in range(0, n - train_size - embargo - test_size + 1, test_size):
        fold += 1

        train_start = start
        train_end = start + train_size

        test_start = train_end + embargo
        test_end = test_start + test_size

        train_df = df_model.iloc[train_start:train_end].copy()
        test_df = df_model.iloc[test_start:test_end].copy()

        # split training into primary-train and meta-train
        split_idx = int(len(train_df) * (1 - META_TRAIN_FRAC))

        primary_train_df = train_df.iloc[:split_idx].copy()
        meta_train_df = train_df.iloc[split_idx:].copy()

        X_primary = primary_train_df[feature_cols]
        y_primary = primary_train_df[target_col].astype(int)

        X_meta_raw = meta_train_df[feature_cols]
        y_meta_true = meta_train_df[target_col].astype(int)

        X_test_raw = test_df[feature_cols]
        y_test = test_df[target_col].astype(int)

        if y_primary.nunique() < 2 or y_test.nunique() < 2:
            print(f"Fold {fold}: skipped one-class train/test.")
            continue

        # ======================
        # PCA preprocessing
        # ======================
        scaler = StandardScaler()
        pca = PCA(n_components=None)

        X_primary_scaled = scaler.fit_transform(X_primary)
        X_primary_pca = pca.fit_transform(X_primary_scaled)

        X_meta_pca = pca.transform(scaler.transform(X_meta_raw))
        X_test_pca = pca.transform(scaler.transform(X_test_raw))

        pca_cols = [f"PC{i+1}" for i in range(X_primary_pca.shape[1])]

        X_primary_pca = pd.DataFrame(X_primary_pca, columns=pca_cols)
        X_meta_pca = pd.DataFrame(X_meta_pca, columns=pca_cols)
        X_test_pca = pd.DataFrame(X_test_pca, columns=pca_cols)

        # ======================
        # Primary XGBoost model
        # ======================
        primary_model = xgb.XGBClassifier(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=30,
            n_jobs=-1,
        )

        primary_model.fit(X_primary_pca, y_primary)

        meta_primary_proba = primary_model.predict_proba(X_meta_pca)[:, 1]
        test_primary_proba = primary_model.predict_proba(X_test_pca)[:, 1]

        meta_primary_signal = (meta_primary_proba >= PRIMARY_THRESHOLD).astype(int)
        test_primary_signal = (test_primary_proba >= PRIMARY_THRESHOLD).astype(int)

        # ======================
        # Meta-labeling
        # Only train meta-model on trades proposed by primary model
        # ======================
        meta_candidates = meta_primary_signal == 1

        if meta_candidates.sum() < 20 or y_meta_true[meta_candidates].nunique() < 2:
            print(f"Fold {fold}: meta-model skipped, not enough valid meta-label data.")
            test_meta_signal = test_primary_signal.copy()
            meta_used = False

        else:
            X_meta_train = X_meta_pca.loc[meta_candidates].copy()
            X_meta_train["primary_proba"] = meta_primary_proba[meta_candidates]

            y_meta_label = y_meta_true.loc[meta_candidates].astype(int)

            meta_model = xgb.XGBClassifier(
                n_estimators=200,
                max_depth=2,
                learning_rate=0.03,
                subsample=0.8,
                colsample_bytree=0.8,
                eval_metric="logloss",
                random_state=31,
                n_jobs=-1,
            )

            meta_model.fit(X_meta_train, y_meta_label)

            test_candidates = test_primary_signal == 1

            test_meta_signal = np.zeros_like(test_primary_signal)

            if test_candidates.sum() > 0:
                X_test_meta = X_test_pca.loc[test_candidates].copy()
                X_test_meta["primary_proba"] = test_primary_proba[test_candidates]

                test_meta_proba = meta_model.predict_proba(X_test_meta)[:, 1]

                accepted = test_meta_proba >= META_THRESHOLD

                candidate_indices = np.where(test_candidates)[0]
                accepted_indices = candidate_indices[accepted]

                test_meta_signal[accepted_indices] = 1

            meta_used = True

        fold_df = pd.DataFrame({
            "fold": fold,
            "y_true": y_test.values,
            "primary_proba": test_primary_proba,
            "primary_signal": test_primary_signal,
            "meta_signal": test_meta_signal,
            "meta_used": meta_used,
        })

        all_rows.append(fold_df)

        fold_reports.append({
            "fold": fold,
            "primary_signals": int(test_primary_signal.sum()),
            "meta_signals": int(test_meta_signal.sum()),
            "meta_used": meta_used,
            "pca_components": X_primary_pca.shape[1],
            "pca_explained_variance_total": pca.explained_variance_ratio_.sum(),
        })

    return pd.concat(all_rows, ignore_index=True), pd.DataFrame(fold_reports)